# XAI Evaluation — Dataset 3
Quantitative evaluation of **SHAP** and **LIME** across **three models** (RF, SVM, MLP) using:

1. **Correctness**: Incremental Deletion, via AOPC
2. **Consistency**: Jaccard Similarity
3. **Contrastivity**: Local Lipschitz Constants

Each metric is applied identically to all three models

In [4]:
import joblib
import numpy as np
from numpy.linalg import norm
import pandas as pd
import shap
import lime
import lime.lime_tabular
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
import os
from sklearn.metrics import f1_score
from scipy.stats import entropy

warnings.filterwarnings('ignore')

X_train_raw = np.load('../models/ds3/X_train.npy')
X_test_raw = np.load('../models/ds3/X_test.npy')
y_test = np.load('../models/ds3/y_test.npy')
feature_names = pd.read_csv('../models/ds3/feature_names.csv').squeeze().tolist()

X_train = pd.DataFrame(X_train_raw, columns=feature_names)
X_test = pd.DataFrame(X_test_raw,  columns=feature_names)

class_names = ['Healthy', 'Benign', 'Cancer']
n_features = len(feature_names)
train_means = X_train.mean()   

rf_model = joblib.load('../models/ds3/ds3_random_forest.pkl')
svm_model = joblib.load('../models/ds3/ds3_svm.pkl')
mlp_model = joblib.load('../models/ds3/ds3_mlp.pkl')

models = {
    'Random Forest': rf_model,
    'SVM':           svm_model,
    'MLP':           mlp_model,
}

os.makedirs('../plots/ds3/evaluation', exist_ok=True)
#120 samples
#16 features
#Classes: {1: 51, 2: 43, 3: 26}

## SHAP values for all models

In [ ]:
rf_explainer   = shap.TreeExplainer(rf_model)
shap_rf        = rf_explainer.shap_values(X_test)   

background = shap.sample(X_train, 100, random_state=42)
svm_explainer  = shap.KernelExplainer(svm_model.predict_proba, background)
shap_svm       = svm_explainer.shap_values(X_test)

mlp_explainer  = shap.KernelExplainer(mlp_model.predict_proba, background)
shap_mlp       = mlp_explainer.shap_values(X_test)

shap_values_all = {
    'Random Forest': shap_rf,
    'SVM': shap_svm,
    'MLP': shap_mlp
}
print("SHAP values computed for all models.")